# FrozenLake

Une grille 4x4 où l'agent part du coin haut-gauche (S) et doit atteindre le coin bas-droite (G) sans tomber dans les trous (H). 16 états (les 16 cases), 4 actions (haut, bas, gauche, droite). L'environnement est glissant par défaut : l'action choisie n'est pas toujours celle exécutée. La récompense est simple : 1.0 si tu atteins G, 0 sinon.

## Étape 1 — Environnement et Q-table

Création de l'environnement FrozenLake-v1 et initialisation de la Q-table à zéros (16 états × 4 actions).

In [1]:
import gymnasium as gym
import numpy as np

env = gym.make(
    'FrozenLake-v1',
    desc=None,
    map_name="4x4",
    is_slippery=True,
)

n_states  = env.observation_space.n
n_actions = env.action_space.n

# Initialiser la Q-table à zéro
q_table = np.zeros((n_states, n_actions))

print(f"États : {env.observation_space.n}, Actions : {env.action_space.n}")
env.close()

print(q_table)


États : 16, Actions : 4
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


## Étape 2 — Entraînement Q-Learning

### Hyperparamètres

- `n_episodes` : nombre total d'épisodes d'entraînement
- `learning_rate` : vitesse à laquelle l'agent met à jour ses valeurs Q (0 = n'apprend rien, 1 = oublie tout ce qu'il savait)
- `gamma` : importance des récompenses futures (proche de 1 = l'agent planifie loin, proche de 0 = il ne pense qu'à la récompense immédiate)
- `epsilon` : probabilité d'explorer (choisir une action aléatoire) plutôt qu'exploiter la Q-table
- `epsilon_min` : plancher d'epsilon, l'agent garde toujours une petite part d'exploration
- `epsilon_decay` : facteur de réduction d'epsilon après chaque épisode

In [2]:
n_episodes = 10000
learning_rate = 0.8
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995

### Boucle d'entraînement

**Epsilon-greedy** : à chaque pas, l'agent tire un nombre aléatoire entre 0 et 1. S'il est inférieur à epsilon, il explore (action aléatoire). Sinon, il exploite (meilleure action connue dans la Q-table). Au début epsilon est élevé donc l'agent explore beaucoup, il diminue au fil des épisodes pour laisser place à l'exploitation.

**Équation de Bellman** : formule qui met à jour la Q-table après chaque action.

```
Q[état, action] = Q[état, action] + lr × (récompense + gamma × max(Q[état_suivant]) − Q[état, action])
```

En clair : on ajuste la valeur actuelle en direction de ce qu'on vient d'observer (récompense immédiate + meilleure valeur future estimée). Le `learning_rate` contrôle à quelle vitesse on fait cet ajustement.

In [3]:
env = gym.make(
    'FrozenLake-v1',
    desc=None,
    map_name="4x4",
    is_slippery=True,
    render_mode=None
)


for i in range(n_episodes):
    state, _ = env.reset()
    episode_over = False
    while not episode_over:
        # Exploration
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()   
        else: # Exploitation
            action = np.argmax(q_table[state])   

        new_state, reward, terminated, truncated, info = env.step(action)

        episode_over = terminated or truncated

        best_next = np.max(q_table[new_state])
        q_table[state, action] += learning_rate * (
            reward + gamma * best_next - q_table[state, action]
        )
        state = new_state

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    

print(q_table)



[[1.47145354e-01 2.28428362e-01 5.42576832e-02 5.35606244e-02]
 [6.28198756e-03 9.28455675e-03 5.76357014e-03 1.31730848e-01]
 [5.56460932e-03 1.06012546e-02 1.33140143e-02 4.66082434e-02]
 [5.35354405e-03 9.82229052e-03 2.10243258e-03 2.15618511e-02]
 [5.02839019e-01 1.56720851e-02 6.63639340e-03 3.01622046e-02]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [8.12282589e-07 3.06807749e-08 7.25587379e-03 2.09902969e-07]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [3.85301332e-02 2.67845599e-02 1.70740706e-02 2.75198058e-01]
 [7.41169555e-03 7.84732938e-02 3.26041121e-02 4.71433178e-03]
 [3.64153123e-02 1.06029256e-03 5.56460952e-03 1.29692484e-04]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [1.80849974e-01 8.20631869e-02 9.29040125e-01 1.11423194e-01]
 [6.99453904e-01 9.99986779e-01 1.72544910e-01 1.37668944e-01]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.000000

## Étape 3 — Évaluation

Pure exploitation de la Q-table apprise (pas d'exploration). On mesure le taux de réussite sur 100 épisodes.

In [4]:
env = gym.make(
    'FrozenLake-v1',
    desc=None,
    map_name="4x4",
    is_slippery=True,
    render_mode=None
)

n_episodes_final = 100
total_win = 0
for i in range(n_episodes_final):
    state, _ = env.reset()
    episode_over = False
    while not episode_over:
        action = np.argmax(q_table[state])   

        new_state, reward, terminated, truncated, info = env.step(action)

        episode_over = terminated or truncated

        if episode_over and reward == 1:
            total_win += 1

        state = new_state    

print(f"Taux de réussite sur {n_episodes_final} épisodes : {total_win / n_episodes_final * 100}%")



Taux de réussite sur 100 épisodes : 54.0%
